← [Dos formas de saber](02-dos-formas-de-saber.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Factor de certeza](04-factor-de-certeza.ipynb) →

# 03 · Reglas y hechos

Un sistema experto tiene tres partes. Conviene separarlas mentalmente porque en
el código son tres archivos distintos:

| Parte | Qué es | Archivo |
| --- | --- | --- |
| Base de conocimiento | Lo que el sistema sabe, siempre | `knowledge_base.py` |
| Memoria de trabajo | Lo que sabe del objeto que tiene enfrente, ahora | `working_memory.py` |
| Motor de inferencia | Cómo combina lo uno con lo otro | `inference_engine.py` |

La analogía: la base de conocimiento es el libro de texto, la memoria de
trabajo es la hoja de examen que se está resolviendo, y el motor de inferencia
es el estudiante.



## Los atributos: el vocabulario

Antes de escribir reglas hay que fijar de qué se puede hablar. Reci define
**nueve atributos**, cada uno con valores cerrados (`knowledge_base.py:9`):

| Atributo | Valores posibles |
| --- | --- |
| `transparencia` | alta, media, baja, ninguna |
| `color` | transparente, ámbar, verde oscuro, blanco opaco, negro, variado vivo, marrón tierra, metálico |
| `forma` | cilíndrica delgada, cilíndrica estándar, cilíndrica ancha, cónica, rectangular plana, irregular |
| `brillo` | alto nítido, medio difuso, bajo, metálico |
| `tapa` | rosca plástico, corona metálica, twist-off metálica, tapa ancha metálica, domo plástico, sin tapa, sellado |
| `textura` | lisa brillante, lisa sin brillo, rugosa, fibrosa |
| `rigidez` | rígido, flexible, indefinido |
| `confianza_ml` | alta, media, baja |
| `objeto_reconocido` | ~35 objetos concretos, más `desconocido` |

Que los valores sean cerrados importa: `validator.py` rechaza cualquier hecho
con un valor fuera de la lista. Si el proveedor de visión inventa un color
—cosa que un modelo de lenguaje puede hacer— el sistema lo detecta en vez de
razonar sobre un dato inválido.

Nota el atributo `confianza_ml`: el sistema experto **recibe como dato** lo
segura que está la parte de aprendizaje automático. Las dos familias del
documento anterior no están del todo aisladas.



## Una regla

Cada regla es un `SI … ENTONCES` con un peso. Del código
(`knowledge_base.py:102`):

```python
Rule("R01",
     {"objeto_reconocido": "botella_mocachino", "confianza_ml": "alta"},
     "VIDRIO",
     0.98,
     "Botella de mocachino identificada con alta confianza — es vidrio ámbar pequeño")
```

Sus cinco partes:

| Posición | Nombre | Valor aquí |
| --- | --- | --- |
| 1 | Identificador | `R01` |
| 2 | **Antecedente** (el SI) | objeto = mocachino **Y** confianza = alta |
| 3 | **Consecuente** (el ENTONCES) | `VIDRIO` |
| 4 | **Factor de certeza** | `0.98` |
| 5 | Justificación legible | el texto final |

El antecedente es un diccionario, y todas sus condiciones deben cumplirse: es
una **conjunción** (Y), nunca una disyunción (O). Para expresar "o" se escriben
dos reglas. Por eso `R01` tiene un hermano `R01_B` con `confianza_ml: "media"` y
un CF menor (0,88): mismo objeto, menos seguridad, menos peso.

Ese quinto campo —la justificación— no afecta el cálculo. Existe para que el
sistema pueda explicarse, que es la ventaja de esta familia
([02](02-dos-formas-de-saber.ipynb)).



## Las 193 reglas

No es un número redondo por casualidad: creció caso por caso, cada vez que un
objeto real del campus se clasificó mal. Cubren:

- objetos concretos (`botella_gatorade`, `botella_pony_malta`, `vaso_kraft`);
- combinaciones de atributos genéricos, para lo no catalogado;
- materiales que no abren compuerta (`ORGANICO`, `LATA`) — se reconocen
  explícitamente para poder **rechazarlos** con confianza, en vez de forzarlos
  a vidrio o plástico.

Esa última categoría es una ventaja directa sobre la red neuronal binaria, que
no puede rechazar nada: sobre 23 objetos que no van en ninguna compuerta, el
modelo local rechazó 0 y el sistema experto 20.



## La memoria de trabajo

Mientras la base de conocimiento no cambia nunca, la memoria de trabajo se
llena y se vacía con cada objeto (`working_memory.py`):

```python
memoria.agregar_hecho("transparencia", "alta", fuente="ml")
memoria.agregar_hecho("tapa", "corona_metalica", fuente="ml")
```

Dos detalles del diseño:

**Cada hecho registra su fuente** (`ml`, `sensor`, `inferencia`). Permite
distinguir lo observado de lo deducido — y dejaría entrar un sensor físico
como una fuente más, sin cambiar el motor.

**Se guarda un historial de cambios.** No solo el estado final, sino cómo se
llegó a él. Es lo que alimenta el reporte de explicación.



## Qué falta

Ya tenemos reglas con pesos y hechos sobre los que aplicarlas. Faltan dos cosas:

1. Qué pasa cuando **varias reglas** concluyen lo mismo con pesos distintos.
   → [Factor de certeza](04-factor-de-certeza.ipynb)
2. En qué **orden** se aplican y cómo se detectan contradicciones.
   → [Motor de inferencia](05-motor-de-inferencia.ipynb)

---

← [Dos formas de saber](02-dos-formas-de-saber.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Factor de certeza](04-factor-de-certeza.ipynb) →
